# Module 17 — Scraping

Module 16 fetched a file somebody had prepared for a program. This module is what you
do when nobody prepared one: the numbers are in a table on a page, and the page was
written for a person.

Two things run through it. The technical one is that **an HTML parser never fails** —
it repairs, silently, and section 5 shows it repairing a page into data that is
plausible and wrong. That is the third time this course has met that shape, after
`latin-1` in module 08 and the missing charset in module 16.

The other is that scraping is the one technique in this course with someone else on
the receiving end. Section 7 is about that, and it is not a footnote.

`server.py` next to this notebook serves the pages, so no network is involved.

In [ ]:
import sys
from pathlib import Path

HERE = Path.cwd() if (Path.cwd() / "server.py").is_file() else Path.cwd() / "17_scraping"
sys.path.append(str(HERE))  # so that `import server` finds it -- module 10

import requests  # noqa: E402
from bs4 import BeautifulSoup  # noqa: E402
from server import serve  # noqa: E402

with serve() as base:
    response = requests.get(f"{base}/readings.html", timeout=5)
    response.raise_for_status()

print(response.headers["Content-Type"])
print(response.text[:80])

## 1. A parsed page is a tree

`BeautifulSoup(html, "html.parser")` turns the text into a tree you can walk. The
second argument is the parser, and it is not optional in practice — leave it out and
you get a warning plus whatever is installed, which means the same code behaves
differently on two machines.

In [ ]:
with serve() as base:
    soup = BeautifulSoup(requests.get(f"{base}/readings.html", timeout=5).text, "html.parser")

print(soup.title.string)
print(soup.h1.get_text())
print(soup.find("td").get_text())  # the first one
print(len(soup.find_all("td")))  # all of them

`&mdash;` in the source came out as `—` and `&deg;` as `°`: the parser resolves
entities, so what you get is text rather than markup.

Two families of method, and it is worth picking one:

| | |
| --- | --- |
| `find(...)` / `find_all(...)` | by tag name and attributes, as keyword arguments |
| `select_one(...)` / `select(...)` | by **CSS selector**, as a string |

The CSS form is usually shorter and reads the way the page's own stylesheet does,
which is a real advantage when you are looking at the page in a browser's inspector
while you write the selector.

In [ ]:
with serve() as base:
    soup = BeautifulSoup(requests.get(f"{base}/readings.html", timeout=5).text, "html.parser")

print([td.get_text() for td in soup.select("tr.row td.value")])  # descendant, by class
print(soup.select_one("#readings").get("class"))  # by id
print(len(soup.select("table.data tr")))  # header row included
print(soup.select_one("a.next")["href"])  # an attribute

Note `.get("class")` gave a **list**. An HTML element can have several classes, so
Beautiful Soup gives you all of them; `class="row fault"` is `["row", "fault"]`, and
code that treats it as a string works right up to the first element with two.

In [ ]:
with serve() as base:
    soup = BeautifulSoup(requests.get(f"{base}/readings.html", timeout=5).text, "html.parser")

for row in soup.select("tr.row"):
    print(row.get("class"))

## 2. What happens when it is not there

This is the part that decides whether your scraper fails usefully or fails
mysteriously. **`find` and `select_one` return `None`**, they do not raise. So the
error arrives one line later, in a message about `NoneType`.

In [ ]:
with serve() as base:
    soup = BeautifulSoup(requests.get(f"{base}/readings.html", timeout=5).text, "html.parser")

missing = soup.find("h2")  # there is no h2 on the page

# What is `missing`, and what does the next line do?
assert missing is ...

try:
    missing.get_text()
    outcome = "worked"
except Exception as err:
    outcome = type(err).__name__

assert outcome == ...

`AttributeError: 'NoneType' object has no attribute 'get_text'` is the single most
common scraping error, and its message names neither the selector nor the page. When
you meet it, the question is always the same: **which selector matched nothing, and
why** — the page changed, the content is loaded by JavaScript, or the selector was
wrong from the start.

An attribute that is not there behaves differently again: `element["href"]` raises
`KeyError`, `element.get("href")` returns `None`.

In [ ]:
with serve() as base:
    soup = BeautifulSoup(requests.get(f"{base}/readings.html", timeout=5).text, "html.parser")

paragraph = soup.select_one("h1")

print(paragraph.get("href"))  # None
print(paragraph.get("href", "no link"))  # a default, as on a dict

try:
    paragraph["href"]
except KeyError as err:
    print("KeyError:", err)

Which suggests the shape a scraper should have: **check the selector, once, and say
what went wrong.**

```python
table = soup.select_one("#readings")
if table is None:
    raise ScrapeError("no #readings table -- has the page changed?")
```

That is module 09's exception class of your own, and it is the difference between a
report you can act on and an `AttributeError` at three in the morning.

## 3. A table into records

The everyday job. One row at a time, cells by selector, and `strip=True` because HTML
is full of whitespace that is invisible in a browser.

In [ ]:
with serve() as base:
    soup = BeautifulSoup(requests.get(f"{base}/readings.html", timeout=5).text, "html.parser")

rows = []
for row in soup.select("tr.row"):
    cells = [cell.get_text(strip=True) for cell in row.select("td")]
    rows.append(cells)

for cells in rows:
    print(cells)

print(type(rows[0][1]).__name__)  # everything scraped is a str -- module 08 again

Everything that comes out of a page is a **string**, exactly as with a CSV. `21.7` is
text until you convert it, and the conversion is where bad data announces itself —
which is module 09's `float()` in a `try`.

A dict per row reads better than a list, and it survives a column being inserted:

In [ ]:
with serve() as base:
    soup = BeautifulSoup(requests.get(f"{base}/readings.html", timeout=5).text, "html.parser")

records = []
for row in soup.select("tr.row"):
    records.append(
        {
            "tag": row.select_one("td.tag").get_text(strip=True),
            "value": float(row.select_one("td.value").get_text(strip=True)),
            "fault": "fault" in row.get("class", []),
        }
    )

for record in records:
    print(record)

print([r["tag"] for r in records if r["fault"]])

Selecting by class (`td.tag`, `td.value`) rather than by position (`cells[0]`,
`cells[1]`) is the choice that survives a column being added. When the page has no
useful classes you have no option, and then the scraper is as fragile as the page's
layout — which is worth knowing before you promise anybody it will keep working.

## 4. Relative links

`href="/page2.html"` is not a URL you can request. `urllib.parse.urljoin` resolves it
against the page it came from, and it handles all three cases correctly.

In [ ]:
from urllib.parse import urljoin

page = "http://example.invalid/a/b.html"

print(urljoin(page, "/next"))  # from the root
print(urljoin(page, "next"))  # from the current directory
print(urljoin(page, "https://elsewhere.invalid/x"))  # already absolute: unchanged

In [ ]:
with serve() as base:
    page_url = f"{base}/readings.html"
    soup = BeautifulSoup(requests.get(page_url, timeout=5).text, "html.parser")

    next_href = soup.select_one("a.next")["href"]
    next_url = urljoin(page_url, next_href)

    print(next_href, "->", next_url)

    second = BeautifulSoup(requests.get(next_url, timeout=5).text, "html.parser")
    print(second.h1.get_text())
    print([td.get_text(strip=True) for td in second.select("td.tag")])

## 5. The parser never tells you the page was broken

Now the section this module is for. `/sloppy.html` has the same three readings as
`/readings.html`, with the closing `</td>` and `</tr>` tags left out — which is legal
HTML and is what a great many real pages look like.

Predict how many cells the first row has.

In [ ]:
with serve() as base:
    sloppy = BeautifulSoup(requests.get(f"{base}/sloppy.html", timeout=5).text, "html.parser")

first_row = sloppy.select("tr.row")[0]

# The row has two readings in it: a tag and a value. How many `td` does it have?
assert len(first_row.select("td")) == ...

In [ ]:
with serve() as base:
    good = BeautifulSoup(requests.get(f"{base}/readings.html", timeout=5).text, "html.parser")
    sloppy = BeautifulSoup(requests.get(f"{base}/sloppy.html", timeout=5).text, "html.parser")

print("well formed:")
for row in good.select("tr.row"):
    print("  ", [c.get_text(strip=True) for c in row.select("td")])

print("the same data, tags unclosed:")
for row in sloppy.select("tr.row"):
    print("  ", [c.get_text(strip=True) for c in row.select("td")])

Read that output.

`html.parser` could not tell where each `<td>` ended, so it **nested** them: each cell
contains everything after it. The first row reports four cells instead of two, and the
first cell's text is `TH-0121.7TH-0491.0` — every value on the page, concatenated.

No exception. No warning. A `len()` that looks like a number of columns. If your code
takes `cells[0]` and `cells[1]`, it gets two strings of the right type and the wrong
content, and the mistake reaches your database.

**That is the same failure as `latin-1` in module 08 and the missing charset in module
16**, for the third and last time: the tool that cannot fail is the one that hurts
you. A parser that raised on malformed HTML would be useless — most of the web is
malformed — so it guesses, and the guess is your problem.

Two consequences worth carrying:

- **Name the parser.** `html.parser` is the standard library's and needs nothing
  installed. `lxml` is faster and repairs differently; `html5lib` repairs the way a
  browser does, slowly. Different parsers give **different answers on broken input**,
  so an unnamed parser makes your result depend on what happens to be installed.
- **Check the shape, do not trust it.** If a row should have three cells, say so:

```python
cells = row.select("td")
if len(cells) != 3:
    raise ScrapeError(f"row has {len(cells)} cells, expected 3")
```

That one `if` turns section 5's silent corruption into a message. It is the cheapest
line in any scraper.

## 6. What a browser shows and what you get are not the same page

Everything above assumed the numbers are in the HTML that arrived. Increasingly they
are not: the page ships a script, the script fetches JSON, and the table is built in
the browser. `requests` runs no JavaScript, so `soup.select("tr.row")` finds nothing
and you get section 2's `AttributeError`.

How to tell, before writing a scraper: fetch the page and look for the value you want
in `response.text`.

In [ ]:
with serve() as base:
    text = requests.get(f"{base}/readings.html", timeout=5).text

print("21.7" in text)  # in the HTML: scraping will work
print("view-source is the truth; the inspector shows the page after JavaScript")

If the value is not in the text, the options in order of preference:

1. **Find the API the page itself uses.** Open the browser's network tab, watch the
   requests it makes, and call that endpoint directly with module 16. It usually
   returns JSON, which is better than HTML in every way — typed, stable, documented
   by its shape.
2. **Look for a published API.** Often there is one, and it is often permitted where
   scraping is not.
3. **Drive a real browser** — Playwright or Selenium. Works, and costs you a browser
   per scrape plus a much more fragile setup.

Option 1 is worth trying first almost every time, and it is a good habit generally:
the question "where does this page get its data?" often has a much better answer than
the page itself.

## 7. There is somebody on the other end

Every other technique in this course affects only your own machine. This one sends
requests to a server somebody pays for. That makes it the one place where "it works"
is not the whole of the question.

**`robots.txt`** is where a site states what automated clients may fetch. The standard
library parses it, so there is no reason to guess.

In [ ]:
from urllib.robotparser import RobotFileParser

with serve() as base:
    rules = RobotFileParser()
    rules.set_url(f"{base}/robots.txt")
    rules.read()

    print(rules.can_fetch("*", f"{base}/readings.html"))
    print(rules.can_fetch("*", f"{base}/private/secret"))
    print(rules.crawl_delay("*"), "second between requests")

Read that last line: the site asked for a delay, and it is asking politely for the
thing that decides whether your scraper is a nuisance.

One detail that catches people, and it is measurable: `Disallow: /admin` matches by
**prefix**, not by path segment. So it forbids `/admin`, `/admin/users` **and**
`/administration`.

In [ ]:
from urllib.robotparser import RobotFileParser

with serve() as base:
    rules = RobotFileParser()
    rules.set_url(f"{base}/robots.txt")
    rules.read()

    for path in ("/admin", "/admin/users", "/administration", "/adminfoo"):
        print(f"{path:18} {rules.can_fetch('*', base + path)}")

And `robots.txt` can name specific clients — the file this server sends forbids
`GreedyBot` everything, which only works if the client identifies itself honestly:

In [ ]:
from urllib.robotparser import RobotFileParser

with serve() as base:
    rules = RobotFileParser()
    rules.set_url(f"{base}/robots.txt")
    rules.read()

    print(rules.can_fetch("*", f"{base}/readings.html"))
    print(rules.can_fetch("GreedyBot", f"{base}/readings.html"))

print(requests.utils.default_headers()["User-Agent"])  # what you send unless you say

`python-requests/2.34.2` is what a site sees by default. Setting a `User-Agent` that
says who you are and how to reach you is the courteous thing and the self-interested
thing: an operator who can email you will do that before blocking your address range.

```python
HEADERS = {"User-Agent": "sensor-coursework/1.0 (student project; you@example.org)"}
```

**Faking a browser's User-Agent to get past a block is a different act**, and the
thing to notice is that it is a decision rather than a technique. The block was a
statement.

### The rules that are not about `robots.txt`

`robots.txt` is a request, not a law, and the law is elsewhere. Four things worth
knowing before you scrape anything that is not a coursework fixture:

- **The terms of service** may forbid automated access whatever `robots.txt` says.
  They are a contract, and courts have treated them as one.
- **Copyright applies to the content.** Fetching a page to compute a statistic is a
  different act from republishing its text, and the second one is the one with a
  rights holder.
- **Personal data is regulated.** Under the GDPR, scraping names, addresses,
  photographs or anything that identifies a person is processing personal data, and
  needs a legal basis — which "it was publicly visible" is not, on its own.
- **Load is a cost you impose.** A loop with no delay against a small site is a
  denial-of-service attack that you did not mean, and it looks identical from the
  server's side.

The practical minimum, and none of it is expensive:

1. Read `robots.txt` and honour it, including `Crawl-delay`.
2. Identify yourself in the `User-Agent`.
3. Sleep between requests — `time.sleep(1)` is fine and one second is not slow.
4. Cache what you fetch, so a bug in your parser does not cost the site a second
   crawl.
5. Fetch what you need, once. Not the whole site because it was easier to write.

Point 4 is the one that also saves *you* time: parse from a local copy while you are
still getting the selectors right, and the site sees one request instead of forty.

---

`exercises/` is next: `exercise_01.py` to `exercise_06.py`, `exercise_09.py`, and two
in `thinking.md` with nothing to run. Exercise 09 crawls both pages, with a delay and
a `robots.txt` check, which is the whole module in one file.

Module 18 takes the scraped rows and asks questions of them — and is where a notebook
finally earns its keep.